In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir('/content/drive/MyDrive/ml_v2/')

Mounted at /content/drive/


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import json

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, BatchNormalization, Layer
)
from tensorflow.keras.callbacks import EarlyStopping

# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 1.5 反轉反向指標
# 原本:
# 分數越高 -> 越不漂綠 / 風險越低
# 反轉後:
# 分數越高 -> 越漂綠 / 風險越高
# 保留原欄位，新增 risk 欄位
# =====================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =====================================
# 3. feature groups (LLaMA)
# Vagueness / Deflection 改為反向後的 risk 指標
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 7. 輕量 grid（只跑3組）
# =====================================
param_configs = [
    {"filters": 32, "dropout_rate": 0.2},
    {"filters": 64, "dropout_rate": 0.2},
    {"filters": 32, "dropout_rate": 0.3},
]

# =====================================
# 8. 自訂 Attention Pooling
# =====================================
class AttentionPooling(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = Dense(1)

    def call(self, inputs):
        # inputs shape: (batch, timesteps, channels)
        scores = self.score_dense(inputs)                  # (batch, timesteps, 1)
        weights = tf.nn.softmax(scores, axis=1)           # (batch, timesteps, 1)
        context = tf.reduce_sum(inputs * weights, axis=1) # (batch, channels)
        return context

# =====================================
# 9. 建 CNN + Attention
# =====================================
def build_cnn_attention(input_length, filters=32, dropout_rate=0.2):
    inputs = Input(shape=(input_length, 1))

    x = Conv1D(filters=filters, kernel_size=2, activation="relu", padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    x = Conv1D(filters=filters * 2, kernel_size=2, activation="relu", padding="same")(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    x = AttentionPooling()(x)

    x = Dense(32, activation="relu")(x)
    x = Dropout(dropout_rate)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =====================================
# 10. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# =====================================
# 11. 從 outer train 再切 validation（group-aware）
# =====================================
def make_group_validation_split(X_train_df, y_train_s, groups_train_s):
    inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(inner_splitter.split(X_train_df, y_train_s, groups=groups_train_s))

    X_tr = X_train_df.iloc[tr_idx]
    X_val = X_train_df.iloc[val_idx]
    y_tr = y_train_s.iloc[tr_idx]
    y_val = y_train_s.iloc[val_idx]
    groups_tr = groups_train_s.iloc[tr_idx]
    groups_val = groups_train_s.iloc[val_idx]

    return X_tr, X_val, y_tr, y_val, groups_tr, groups_val

# =====================================
# 12. 單一 config 評估
# 回傳: summary + fold_df
# =====================================
def evaluate_single_config(X, y, groups, feature_name, config):
    fold_metrics = []
    best_thresholds = []

    print(f"\n=== {feature_name} | config={config} ===")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        # outer train -> train/val（group-aware）
        X_tr_df, X_val_df, y_tr, y_val, _, _ = make_group_validation_split(
            X_train_df, y_train, groups_train
        )

        # imputer
        imputer = SimpleImputer(strategy="median")
        X_tr = imputer.fit_transform(X_tr_df)
        X_val = imputer.transform(X_val_df)
        X_test = imputer.transform(X_test_df)

        # scaler
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        # reshape for CNN
        X_tr = X_tr.reshape(X_tr.shape[0], X_tr.shape[1], 1)
        X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

        # class weight
        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {
            int(cls): float(w) for cls, w in zip(classes, class_weights)
        }

        # build model
        tf.keras.backend.clear_session()
        model = build_cnn_attention(
            input_length=X_tr.shape[1],
            filters=config["filters"],
            dropout_rate=config["dropout_rate"]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=16,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        # threshold tuning
        val_prob = model.predict(X_val, verbose=0).ravel()
        best_threshold, best_val_f1 = find_best_threshold(y_val, val_prob)
        best_thresholds.append(best_threshold)

        # test evaluation
        test_prob = model.predict(X_test, verbose=0).ravel()
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_idx,
            "Model": "CNN + Attention",
            "Feature_Set": feature_name,
            "Config": json.dumps(config, ensure_ascii=False),
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "val_best_f1": best_val_f1
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "CNN + Attention",
        "Feature_Set": feature_name,
        "Config": json.dumps(config, ensure_ascii=False),
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

    return summary, metrics_df

# =====================================
# 13. 每個 feature set 跑 3 組 config
# =====================================
all_results = []
all_fold_results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()

    for config in param_configs:
        summary_result, fold_result_df = evaluate_single_config(
            X, y, groups_all, feature_name, config
        )
        all_results.append(summary_result)
        all_fold_results.append(fold_result_df)

# =====================================
# 14. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "llama_CNN_attention_lightgrid_groupedCV_threshold_reversed_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "llama_CNN_attention_lightgrid_groupedCV_threshold_reversed_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 15. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

best_numeric_cols = best_results_df.select_dtypes(include=[np.number]).columns
best_results_df[best_numeric_cols] = best_results_df[best_numeric_cols].round(4)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "llama_CNN_attention_lightgrid_best_per_featureset_reversed_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 16. 檢查反轉是否成功（可選）
# =====================================
print("\n===== Check reversed indicators =====")
print(df[[
    "llama_vagueness_score_1", "llama_vagueness_risk_1",
    "llama_deflection_score_1", "llama_deflection_risk_1"
]].head())


=== M1: Semantic | config={'filters': 32, 'dropout_rate': 0.2} ===
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.46 | F1=0.8000 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.53 | F1=0.8000 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 3 started


[M1: Semantic] Fold 3 done | Threshold=0.88 | F1=0.6667 | ROC_AUC=0.9688 | PR_AUC=0.8875
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.37 | F1=0.0000 | ROC_AUC=0.5000 | PR_AUC=0.1051
[M1: Semantic] Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.69 | F1=0.8000 | ROC_AUC=0.9493 | PR_AUC=0.8889
[M1: Semantic] Fold 6 started
[M1: Semantic] Fold 6 done | Threshold=0.76 | F1=0.6667 | ROC_AUC=0.8885 | PR_AUC=0.7103
[M1: Semantic] Fold 7 started
[M1: Semantic] Fold 7 done | Threshold=0.22 | F1=0.2500 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 8 started
[M1: Semantic] Fold 8 done | Threshold=0.64 | F1=0.4000 | ROC_AUC=0.9225 | PR_AUC=0.5667
[M1: Semantic] Fold 9 started
[M1: Semantic] Fold 9 done | Threshold=0.90 | F1=0.5000 | ROC_AUC=0.8810 | PR_AUC=0.3595
[M1: Semantic] Fold 10 started
[M1: Semantic] Fold 10 done | Threshold=0.24 | F1=0.1818 | ROC_AUC=0.8243 | PR_AUC=0.1909

=== M1: Semantic | config={'filters': 64, 'dropout_rate': 0.2} ===
[M1: Sem

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import json

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, BatchNormalization, Layer
)
from tensorflow.keras.callbacks import EarlyStopping

# =====================================
# 0. 固定隨機種子
# =====================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 1.5 反轉反向指標
# 原本:
# 分數越高 -> 越不漂綠 / 風險越低
# 反轉後:
# 分數越高 -> 越漂綠 / 風險越高
# 保留原欄位，新增 risk 欄位
# =====================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

# =====================================
# 2. target / groups
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")
if "Company" not in df.columns:
    raise ValueError("Company 不存在於資料中")
if df["Company"].isna().any():
    raise ValueError("Company 欄位有缺值，groups 不能有 NaN")

y = df["label"].astype(int)
groups_all = df["Company"]

# =====================================
# 3. feature groups (chatgpt)
# Vagueness / Deflection 改為反向後的 risk 指標
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. outer CV
# =====================================
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 7. 輕量 grid（只跑3組）
# =====================================
param_configs = [
    {"filters": 32, "dropout_rate": 0.2},
    {"filters": 64, "dropout_rate": 0.2},
    {"filters": 32, "dropout_rate": 0.3},
]

# =====================================
# 8. 自訂 Attention Pooling
# =====================================
class AttentionPooling(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = Dense(1)

    def call(self, inputs):
        # inputs shape: (batch, timesteps, channels)
        scores = self.score_dense(inputs)                  # (batch, timesteps, 1)
        weights = tf.nn.softmax(scores, axis=1)           # (batch, timesteps, 1)
        context = tf.reduce_sum(inputs * weights, axis=1) # (batch, channels)
        return context

# =====================================
# 9. 建 CNN + Attention
# =====================================
def build_cnn_attention(input_length, filters=32, dropout_rate=0.2):
    inputs = Input(shape=(input_length, 1))

    x = Conv1D(filters=filters, kernel_size=2, activation="relu", padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    x = Conv1D(filters=filters * 2, kernel_size=2, activation="relu", padding="same")(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    x = AttentionPooling()(x)

    x = Dense(32, activation="relu")(x)
    x = Dropout(dropout_rate)(x)

    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# =====================================
# 10. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# =====================================
# 11. 從 outer train 再切 validation（group-aware）
# =====================================
def make_group_validation_split(X_train_df, y_train_s, groups_train_s):
    inner_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, val_idx = next(inner_splitter.split(X_train_df, y_train_s, groups=groups_train_s))

    X_tr = X_train_df.iloc[tr_idx]
    X_val = X_train_df.iloc[val_idx]
    y_tr = y_train_s.iloc[tr_idx]
    y_val = y_train_s.iloc[val_idx]
    groups_tr = groups_train_s.iloc[tr_idx]
    groups_val = groups_train_s.iloc[val_idx]

    return X_tr, X_val, y_tr, y_val, groups_tr, groups_val

# =====================================
# 12. 單一 config 評估
# 回傳: summary + fold_df
# =====================================
def evaluate_single_config(X, y, groups, feature_name, config):
    fold_metrics = []
    best_thresholds = []

    print(f"\n=== {feature_name} | config={config} ===")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train_df = X.iloc[train_idx].copy()
        X_test_df = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()
        groups_train = groups.iloc[train_idx].copy()

        # outer train -> train/val（group-aware）
        X_tr_df, X_val_df, y_tr, y_val, _, _ = make_group_validation_split(
            X_train_df, y_train, groups_train
        )

        # imputer
        imputer = SimpleImputer(strategy="median")
        X_tr = imputer.fit_transform(X_tr_df)
        X_val = imputer.transform(X_val_df)
        X_test = imputer.transform(X_test_df)

        # scaler
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        # reshape for CNN
        X_tr = X_tr.reshape(X_tr.shape[0], X_tr.shape[1], 1)
        X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

        # class weight
        classes = np.unique(y_tr)
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_tr
        )
        class_weight_dict = {
            int(cls): float(w) for cls, w in zip(classes, class_weights)
        }

        # build model
        tf.keras.backend.clear_session()
        model = build_cnn_attention(
            input_length=X_tr.shape[1],
            filters=config["filters"],
            dropout_rate=config["dropout_rate"]
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=16,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=0
        )

        # threshold tuning
        val_prob = model.predict(X_val, verbose=0).ravel()
        best_threshold, best_val_f1 = find_best_threshold(y_val, val_prob)
        best_thresholds.append(best_threshold)

        # test evaluation
        test_prob = model.predict(X_test, verbose=0).ravel()
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_idx,
            "Model": "CNN + Attention",
            "Feature_Set": feature_name,
            "Config": json.dumps(config, ensure_ascii=False),
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "val_best_f1": best_val_f1
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "CNN + Attention",
        "Feature_Set": feature_name,
        "Config": json.dumps(config, ensure_ascii=False),
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

    return summary, metrics_df

# =====================================
# 13. 每個 feature set 跑 3 組 config
# =====================================
all_results = []
all_fold_results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()

    for config in param_configs:
        summary_result, fold_result_df = evaluate_single_config(
            X, y, groups_all, feature_name, config
        )
        all_results.append(summary_result)
        all_fold_results.append(fold_result_df)

# =====================================
# 14. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\n===== All Results =====")
print(results_df)

results_df.to_csv(
    "chatgpt_CNN_attention_lightgrid_groupedCV_threshold_reversed_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

folds_df.to_csv(
    "chatgpt_CNN_attention_lightgrid_groupedCV_threshold_reversed_fold_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 15. 每個 feature set 挑 F1 最佳 config
# =====================================
best_results_df = (
    results_df.sort_values(["Feature_Set", "F1_mean"], ascending=[True, False])
    .groupby("Feature_Set", as_index=False)
    .first()
)

best_numeric_cols = best_results_df.select_dtypes(include=[np.number]).columns
best_results_df[best_numeric_cols] = best_results_df[best_numeric_cols].round(4)

print("\n===== Best Config per Feature Set =====")
print(best_results_df)

best_results_df.to_csv(
    "chatgpt_CNN_attention_lightgrid_best_per_featureset_reversed_mean_std.csv",
    index=False,
    encoding="utf-8-sig"
)

# =====================================
# 16. 檢查反轉是否成功（可選）
# =====================================
print("\n===== Check reversed indicators =====")
print(df[[
    "chatgpt_vagueness_score_1", "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1", "chatgpt_deflection_risk_1"
]].head())


=== M1: Semantic | config={'filters': 32, 'dropout_rate': 0.2} ===
[M1: Semantic] Fold 1 started
[M1: Semantic] Fold 1 done | Threshold=0.10 | F1=0.7273 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 2 started
[M1: Semantic] Fold 2 done | Threshold=0.32 | F1=0.5714 | ROC_AUC=0.9571 | PR_AUC=0.7000
[M1: Semantic] Fold 3 started
[M1: Semantic] Fold 3 done | Threshold=0.71 | F1=0.8571 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 4 started
[M1: Semantic] Fold 4 done | Threshold=0.63 | F1=0.2222 | ROC_AUC=0.8600 | PR_AUC=0.6111
[M1: Semantic] Fold 5 started
[M1: Semantic] Fold 5 done | Threshold=0.62 | F1=0.9091 | ROC_AUC=1.0000 | PR_AUC=1.0000
[M1: Semantic] Fold 6 started
[M1: Semantic] Fold 6 done | Threshold=0.75 | F1=0.7500 | ROC_AUC=0.9846 | PR_AUC=0.9267
[M1: Semantic] Fold 7 started
[M1: Semantic] Fold 7 done | Threshold=0.53 | F1=0.0000 | ROC_AUC=0.9412 | PR_AUC=0.3333
[M1: Semantic] Fold 8 started
[M1: Semantic] Fold 8 done | Threshold=0.66 | F1=0.5000 | ROC_AUC=0.9